# 校準式空間 GEV 模擬：完整驗證流程

本 notebook 集中執行：

$$
\text{已知空間真值}
\rightarrow
\text{540 monthly maxima}
\rightarrow
\text{Frozen NN}
\rightarrow
\text{nested buffered Spatial CV}
\rightarrow
\text{FFS + kernel selection}
\rightarrow
\text{OOF parameter recovery}
\rightarrow
RL_{50},RL_{100}.
$$

模型選擇階段只可使用 NN 估計值與候選 predictors；模擬真值只能在 OOF 預測完成後用於評分。

## 0. 執行設定

第一次只想檢查資料時，執行 generation 與 diagnostics 即可。完整 nested Spatial CV 很耗時，確認前段輸出合理後再開啟。

In [ ]:
from pathlib import Path
import sys
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT if (CURRENT / "src").exists() else CURRENT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SIM_DIR = PROJECT_ROOT / "data" / "simulated" / "calibrated_final_model"
CV_DIR = SIM_DIR / "nested_spatial_cv_monthly"
FIGURE_DIR = Path.home() / "Desktop" / "picture"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = PROJECT_ROOT / "results" / "tables"

RUN_GENERATION = False
RUN_DIAGNOSTICS = True
RUN_NESTED_SPATIAL_CV = True
N_REPLICATES = 1
N_YEARS = 45
MONTHS_PER_YEAR = 12
OUTER_FOLDS = 5
INNER_FOLDS = 4
MAX_TRAIN = 800
MIN_TRAIN = 100
MAX_FFS_STEPS = 3
N_JOBS = -2

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SIM_DIR =", SIM_DIR)

## 1. 依真實最終模型生成已知真值

生成式使用真實臺灣本島 GRID、真實候選 predictors，以及真實資料最終選定的 mean structure 與 GP kernel。每個 replicate 重新抽出空間隨機效應與 45 年共 540 個月最大值。

In [ ]:
from calibrated_parametric_simulation import (
    CalibratedSimulationConfig,
    run_calibrated_simulation,
)

if RUN_GENERATION:
    simulation_config = CalibratedSimulationConfig(
        n_years=N_YEARS,
        n_replicates=N_REPLICATES,
        months_per_year=MONTHS_PER_YEAR,
        calibration_max_train=MAX_TRAIN,
        n_restarts_optimizer=0,
        seed=20260820,
    )
    generated_paths = run_calibrated_simulation(
        config=simulation_config,
        output_directory=SIM_DIR,
    )
    display(pd.DataFrame({"generated_file": [str(path) for path in generated_paths]}))
else:
    print("略過重新生成；使用 SIM_DIR 中既有 replicate。")

## 2. 確認 monthly maxima、已知參數與 Frozen NN 輸出

每個 GRID 應有 45 年乘 12 月，共 540 個 monthly maxima。

In [ ]:
MODEL_READY_PATH = SIM_DIR / "replicate_000_model_ready.csv"
MONTHLY_PATH = SIM_DIR / "replicate_000_monthly_maxima.csv"
NN_METRIC_PATH = SIM_DIR / "replicate_000_nn_recovery_metrics.csv"

model_ready = pd.read_csv(MODEL_READY_PATH)
monthly_maxima = pd.read_csv(MONTHLY_PATH)
nn_metrics = pd.read_csv(NN_METRIC_PATH)

summary = pd.DataFrame({
    "item": ["GRID count", "monthly maxima columns", "xi clipped GRID"],
    "value": [
        len(model_ready),
        sum(column.startswith("monthly_max_") for column in monthly_maxima.columns),
        int(model_ready["xi_clipped"].sum()),
    ],
})
display(summary)
display(nn_metrics)

## 3. 模擬生成的臺灣月最大溫資料

這裡展示的是由已知空間 GEV 參數實際抽出的 monthly maxima，不是參數曲面。三個月份使用共同色階，以比較空間位置與月份間的隨機變動。


In [ ]:
from calibrated_simulation_diagnostics import plot_monthly_maxima_examples

monthly_temperature_figure = plot_monthly_maxima_examples(
    model_ready,
    monthly_maxima,
    year_months=((1980, 1), (2002, 7), (2024, 12)),
)
monthly_temperature_path = FIGURE_DIR / "calibrated_simulation_monthly_maxima_examples.png"
monthly_temperature_figure.savefig(
    monthly_temperature_path,
    dpi=220,
    bbox_inches="tight",
)

display(monthly_temperature_figure)
print("Saved:", monthly_temperature_path)


## 4. 檢查模擬曲面是否過度平滑或過度粗糙

比較真實 NN 曲面與模擬真值的共同色階、分布、標準化 variogram 與最近鄰粗糙度。

In [ ]:
from calibrated_simulation_diagnostics import run_diagnostics

if RUN_DIAGNOSTICS:
    diagnostics = run_diagnostics(
        simulation_path=MODEL_READY_PATH,
        figure_directory=FIGURE_DIR,
        table_directory=TABLE_DIR,
    )
    for name in ("surfaces", "distributions", "variograms"):
        display(diagnostics["figures"][name])
    display(diagnostics["roughness"])
else:
    print("略過 diagnostics。")

## 5. 建立 outer geographic folds

Outer folds 只負責最後評估。Outer test fold 與 buffer 內資料不會進入 inner FFS 或 kernel selection。

In [ ]:
from elevation_gp_analysis import prepare_spatial_folds

outer_preview, fold_figure = prepare_spatial_folds(
    model_ready,
    n_folds=OUTER_FOLDS,
    random_state=20260721,
)
fold_figure.savefig(FIGURE_DIR / "calibrated_simulation_outer_folds.png", dpi=220, bbox_inches="tight")
display(fold_figure)
display(
    outer_preview.groupby("spatial_fold")
    .size()
    .rename("n_test_grid")
    .reset_index()
)

## 6. Nested buffered Spatial CV、FFS 與 kernel selection

對每個 outer fold：

1. 保留一區作 outer test。
2. 刪除 test 周圍 target-specific buffer 內的 training GRID。
3. 僅在 outer training 內建立 inner folds。
4. Inner buffered CV 同時執行 grouped FFS 與 RBF/Matérn kernel selection。
5. 用選定模型預測完全未參與選模的 outer test。
6. 五區合併為 OOF predictions。

這一格最耗時。

In [ ]:
from calibrated_simulation_spatial_cv import run_evaluation

if RUN_NESTED_SPATIAL_CV:
    cv_outputs = run_evaluation(
        input_path=MODEL_READY_PATH,
        output_directory=CV_DIR,
        outer_folds=OUTER_FOLDS,
        inner_folds=INNER_FOLDS,
        max_train=MAX_TRAIN,
        min_train=MIN_TRAIN,
        max_steps=MAX_FFS_STEPS,
        min_relative_improvement=0.01,
        maximum_allowed_vif=5.0,
        n_restarts=0,
        n_jobs=N_JOBS,
        random_state=20260721,
    )
else:
    print("尚未重跑 nested Spatial CV；若已有輸出，下一格會直接載入。")

## 7. 載入 nested Spatial CV 結果

In [ ]:
output_files = {
    "predictions": CV_DIR / "calibrated_nested_predictions.csv",
    "selections": CV_DIR / "calibrated_nested_selections.csv",
    "parameter_metrics": CV_DIR / "calibrated_nested_parameter_metrics.csv",
    "return_level_predictions": CV_DIR / "calibrated_nested_return_level_predictions.csv",
    "return_level_metrics": CV_DIR / "calibrated_nested_return_level_metrics.csv",
    "metadata": CV_DIR / "calibrated_nested_metadata.csv",
}

outputs_exist = all(path.exists() for path in output_files.values())
outputs_are_current = outputs_exist and all(
    path.stat().st_mtime >= MODEL_READY_PATH.stat().st_mtime
    for path in output_files.values()
)
if outputs_are_current:
    metadata_check = pd.read_csv(output_files["metadata"])
    current_hash = hashlib.sha256(MODEL_READY_PATH.read_bytes()).hexdigest()
    outputs_are_current = (
        str(metadata_check.loc[0, "input_sha256"]) == current_hash
        and str(metadata_check.loc[0, "block_scale"]).lower() == "monthly"
        and int(metadata_check.loc[0, "n_months"]) == 540
    )

if outputs_are_current:
    cv_outputs = {name: pd.read_csv(path) for name, path in output_files.items()}
    print("已載入既有 nested Spatial CV 輸出。")
else:
    missing = [str(path) for path in output_files.values() if not path.exists()]
    if outputs_exist:
        missing = ["既有 nested Spatial CV 輸出早於新的 monthly simulation，必須重跑。"]
    print("尚缺或輸出已過期；將 RUN_NESTED_SPATIAL_CV 改為 True 並執行第 6 節。")
    display(pd.DataFrame({"missing_file": missing}))

## 8. 模型選回結果

同一 predictor/kernel 若在多個 outer folds 被選回，表示選模較穩定。只看單一 replicate 不足以估計正式選回率。

In [ ]:
if "cv_outputs" in globals() and "selections" in cv_outputs:
    selections = cv_outputs["selections"]
    display(selections)
    selection_frequency = (
        selections.groupby(
            ["target", "selected_groups", "predictors", "kernel", "nu"],
            dropna=False,
        )
        .size()
        .rename("outer_folds_selected")
        .reset_index()
        .sort_values(["target", "outer_folds_selected"], ascending=[True, False])
    )
    display(selection_frequency)

## 9. OOF 參數恢復

主要比較對象是已知模擬真值，不是 NN reference。

In [ ]:
if "cv_outputs" in globals() and "parameter_metrics" in cv_outputs:
    display(cv_outputs["parameter_metrics"])

    predictions = cv_outputs["predictions"].merge(
        model_ready[["station", "x_km", "y_km"]],
        on="station",
        how="left",
        validate="many_to_one",
    )
    targets = [("mu", r"$\mu$"), ("log_sigma", r"$\log\sigma$"), ("xi", r"$\xi$")]
    columns = [
        ("true_value", "Truth"),
        ("nn_value", "Frozen NN"),
        ("oof_prediction", "Nested OOF GP"),
    ]
    fig, axes = plt.subplots(3, 3, figsize=(12, 14), constrained_layout=True)
    for row, (target, label) in enumerate(targets):
        part = predictions.loc[predictions["target"].eq(target)]
        values = np.concatenate([part[column].to_numpy(float) for column, _ in columns])
        vmin, vmax = np.nanquantile(values, [0.01, 0.99])
        for col, (column, title) in enumerate(columns):
            points = axes[row, col].scatter(
                part["x_km"], part["y_km"], c=part[column],
                s=10, cmap="viridis", vmin=vmin, vmax=vmax,
            )
            axes[row, col].set_title(f"{label}: {title}")
            axes[row, col].set_aspect("equal")
            fig.colorbar(points, ax=axes[row, col], shrink=0.75)
    fig.savefig(FIGURE_DIR / "calibrated_simulation_oof_parameter_recovery.png", dpi=220, bbox_inches="tight")
    display(fig)

## 10. OOF return-level 恢復

$$
\widehat{RL}_{T}^{\mathrm{OOF}}
=
RL_T\!\left(
\widehat\mu^{\mathrm{OOF}},
\widehat{\log\sigma}^{\mathrm{OOF}},
\widehat\xi^{\mathrm{OOF}}
\right).
$$

比較 $widehat{RL}_{50}^{\mathrm{OOF}}$、$widehat{RL}_{100}^{\mathrm{OOF}}$ 與生成時已知真值。

In [ ]:
if "cv_outputs" in globals() and "return_level_metrics" in cv_outputs:
    display(cv_outputs["return_level_metrics"])

    rl = cv_outputs["return_level_predictions"].merge(
        model_ready[["station", "x_km", "y_km"]],
        on="station",
        how="left",
        validate="many_to_one",
    )
    fig, axes = plt.subplots(2, 3, figsize=(12, 9), constrained_layout=True)
    columns = [
        ("true_return_level", "Truth"),
        ("nn_return_level", "Frozen NN"),
        ("oof_return_level", "Nested OOF GP"),
    ]
    for row, period in enumerate((50, 100)):
        part = rl.loc[rl["return_period"].eq(period)]
        values = np.concatenate([part[column].to_numpy(float) for column, _ in columns])
        vmin, vmax = np.nanquantile(values, [0.01, 0.99])
        for col, (column, title) in enumerate(columns):
            points = axes[row, col].scatter(
                part["x_km"], part["y_km"], c=part[column],
                s=10, cmap="magma", vmin=vmin, vmax=vmax,
            )
            axes[row, col].set_title(f"RL{period}: {title}")
            axes[row, col].set_aspect("equal")
            fig.colorbar(points, ax=axes[row, col], shrink=0.75)
    fig.savefig(FIGURE_DIR / "calibrated_simulation_oof_return_level_recovery.png", dpi=220, bbox_inches="tight")
    display(fig)

## 11. 結果判讀

完整流程需回答四件事：

- Frozen NN 能否恢復已知 GEV 參數。
- Inner FFS/kernel selection 能否穩定選回生成模型。
- Nested OOF GP 能否恢復未見區域的參數曲面。
- $RL_{50}$ 與 $RL_{100}$ 是否接近已知真值。

單一 replicate 是流程測試。正式論文應使用多個獨立 replicates，報告 RMSE 分布、predictor 選回率與 kernel 選回率。